# Pre-training Phoneme-based BERT for Chinese 

**Mục tiêu của Notebook:**
1. Khởi tạo bộ từ điển hợp nhất (Unified Vocabulary) cho Onset, Rhyme, Tone.
2. Xây dựng Data Collator thực hiện Masked Language Modeling (MLM) ở mức ngữ âm: Che 15% chữ Hán và yêu cầu mô hình dự đoán lại 3 thành phần ngữ âm của chữ đó.
3. Cài đặt kiến trúc `PhonemeBertForMLM` với lớp Embedding dùng chung (Shared Embedding 256-dim).
4. Huấn luyện mô hình từ đầu (From Scratch) bằng PyTorch Native Loop trên Kaggle GPU.

In [ ]:
!nvidia-smi
!pip install -q transformers datasets pypinyin

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel, BertConfig, PreTrainedModel
from datasets import load_dataset
from tqdm.auto import tqdm

# Thiết lập môi trường Kaggle
BASE_DIR = '/kaggle/working'
os.makedirs(BASE_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## Section 1: Load Dataset & Unified Vocabulary Setup

In [ ]:
import os
import sys
import json
from tqdm.auto import tqdm

print("Loading dataset...")

dataset = load_dataset('jed351/Chinese-Common-Crawl-Filtered') 

sys.path.insert(0, '/kaggle/input/zhphoneme')

try:
    from hanzi_processing import HanziProcessor
    processor = HanziProcessor()
    print("Successfully loaded HanziProcessor from /kaggle/input/zhphoneme")
except ImportError:
    raise ImportError("Không tìm thấy hanzi_processing.py! Hãy chắc chắn bạn đã add data zhphoneme vào Kaggle.")

# Khởi tạo các token đặc biệt cho BERT
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]", "<EMPTY>"]
UNIFIED_VOCAB = {token: idx for idx, token in enumerate(SPECIAL_TOKENS)}

vocab_file_path = f"{BASE_DIR}/unified_vocab.json"

# Kiểm tra xem đã có file vocab hợp nhất chưa, nếu chưa thì tự động build
if os.path.exists(vocab_file_path):
    print("Loading existing Unified Vocabulary...")
    with open(vocab_file_path, 'r', encoding='utf-8') as f:
        UNIFIED_VOCAB = json.load(f)
else:
    print("Building Unified Vocabulary from dataset using HanziProcessor...")
    vocab_set = set()
    
    # Lấy mẫu 10,000 câu để quét toàn bộ các ngữ âm có thể xuất hiện
    sample_texts = [example['text'] for example in dataset.select(range(10000))]
    
    for text in tqdm(sample_texts, desc="Extracting IPA components"):
        for char in text:
            # Kiểm tra xem có phải là chữ Hán không (Unicode range)
            if '\u4e00' <= char <= '\u9fff': 
                success, result = processor.process_IPA(char, number_components=3)
                if success:
                    _, (onset, rhyme, tone) = result
                    # Thêm vào set (nếu None thì bỏ qua vì đã có <EMPTY>)
                    if onset: vocab_set.add(onset)
                    if rhyme: vocab_set.add(rhyme)
                    if tone: vocab_set.add(tone)
    
    # Gộp các ngữ âm tìm được vào UNIFIED_VOCAB
    for token in sorted(list(vocab_set)):
        if token not in UNIFIED_VOCAB:
            UNIFIED_VOCAB[token] = len(UNIFIED_VOCAB)
            
    # Lưu lại để các lần chạy sau không phải build lại
    with open(vocab_file_path, 'w', encoding='utf-8') as f:
        json.dump(UNIFIED_VOCAB, f, ensure_ascii=False, indent=2)
    print(f"Saved Unified Vocabulary to {vocab_file_path}")

VOCAB_SIZE = len(UNIFIED_VOCAB)
print(f"Total Unified Vocab Size: {VOCAB_SIZE}")
print(f"ID of [MASK]: {UNIFIED_VOCAB['[MASK]']}")

## Section 2: Phoneme Tokenizer & Dynamic Masking (Data Collator)
Sử dụng PyTorch `Dataset` và `Data Collator` sẽ giúp thực hiện **Dynamic Masking** trực tiếp trên RAM, tiết kiệm ổ cứng và giúp mô hình học được các pattern che ngẫu nhiên khác nhau qua từng Epoch.

In [ ]:
import random

class PhonemeDataset(Dataset):
    def __init__(self, texts, max_length=128):
        self.texts = texts
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]['text']
        # Cắt bớt text để chừa chỗ cho [CLS] và [SEP]
        text = text[:self.max_length - 2] 
        return text

def collate_phoneme_mlm(batch_texts):
    """
    Data Collator thực hiện việc chuyển Text -> Phoneme IDs bằng HanziProcessor 
    và áp dụng Dynamic Masking 15%
    """
    batch_size = len(batch_texts)
    max_len = max(len(t) for t in batch_texts) + 2 # +2 cho CLS và SEP
    
    # Khởi tạo ma trận tensor
    input_ids = torch.full((batch_size, max_len, 3), UNIFIED_VOCAB['[PAD]'], dtype=torch.long)
    labels = torch.full((batch_size, max_len, 3), -100, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_len), dtype=torch.long)
    
    mask_id = UNIFIED_VOCAB['[MASK]']
    unk_id = UNIFIED_VOCAB['[UNK]']
    empty_id = UNIFIED_VOCAB['<EMPTY>']
    
    for i, text in enumerate(batch_texts):
        # 1. Gắn [CLS]
        input_ids[i, 0, :] = UNIFIED_VOCAB['[CLS]']
        attention_mask[i, 0] = 1
        
        for j, char in enumerate(text):
            pos = j + 1
            attention_mask[i, pos] = 1
            
            # 2. Sử dụng HanziProcessor để bóc tách (IPA)
            onset_val, rhyme_val, tone_val = "<EMPTY>", "<EMPTY>", "<EMPTY>"
            
            if '\u4e00' <= char <= '\u9fff':
                success, result = processor.process_IPA(char, number_components=3)
                if success:
                    _, (o, r, t) = result
                    onset_val = o if o else "<EMPTY>"
                    rhyme_val = r if r else "<EMPTY>"
                    tone_val  = t if t else "<EMPTY>"
            else:
                # Fallback cho dấu câu, số, chữ latin
                onset_val = char
                rhyme_val = char
                tone_val  = char
            
            # Map sang ID
            id_onset = UNIFIED_VOCAB.get(onset_val, unk_id)
            id_rhyme = UNIFIED_VOCAB.get(rhyme_val, unk_id)
            id_tone  = UNIFIED_VOCAB.get(tone_val, unk_id)
            
            true_ids = [id_onset, id_rhyme, id_tone]
            
            # 3. Áp dụng MLM (15% bị che)
            # Chỉ che khi nó là chữ Hán (id_onset != unk_id hoặc không phải dấu câu)
            # Ta che bất kỳ token nào không phải special token
            if random.random() < 0.15:
                prob = random.random()
                if prob < 0.8:
                    # 80%: Che bằng [MASK]
                    input_ids[i, pos, :] = mask_id
                elif prob < 0.9:
                    # 10%: Giữ nguyên ID gốc
                    input_ids[i, pos, :] = torch.tensor(true_ids)
                else:
                    # 10%: Thay bằng token ngẫu nhiên trong vocab
                    random_3_ids = torch.randint(6, VOCAB_SIZE, (3,)) # Bắt đầu từ 6 để né Special Tokens
                    input_ids[i, pos, :] = random_3_ids
                
                # Tính Loss cho vị trí này
                labels[i, pos, :] = torch.tensor(true_ids)
            else:
                # 85% giữ nguyên, không tính loss (-100)
                input_ids[i, pos, :] = torch.tensor(true_ids)
                
        # 4. Gắn [SEP]
        seq_end = len(text) + 1
        input_ids[i, seq_end, :] = UNIFIED_VOCAB['[SEP]']
        attention_mask[i, seq_end] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# Tạo DataLoader
train_dataset = PhonemeDataset(dataset, max_length=128)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_phoneme_mlm)

print(f"Total batches per epoch: {len(train_dataloader)}")

## Section 3: Model Architecture (Shared Embeddings & 3 FC Heads)
Ép embedding ngữ âm về kích thước `256` và dùng `torch.cat` để nối thành vector `768` chuẩn bị cho BERT Encoder. Đầu ra sử dụng 1 hàm `CrossEntropyLoss` duy nhất.

In [ ]:
class PhonemeBertConfig(BertConfig):
    def __init__(self, vocab_size=VOCAB_SIZE, phoneme_embed_size=256, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.phoneme_embed_size = phoneme_embed_size
        self.hidden_size = phoneme_embed_size * 3 # 256 * 3 = 768

class PhonemeBertForMLM(PreTrainedModel):
    config_class = PhonemeBertConfig
    
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        
        # SHARED EMBEDDING CHO CẢ 3 THÀNH PHẦN
        self.shared_embeddings = nn.Embedding(
            config.vocab_size, 
            config.phoneme_embed_size, 
            padding_idx=UNIFIED_VOCAB['[PAD]']
        )
        
        # 2. BERT ENCODER
        self.bert = BertModel(config, add_pooling_layer=False)
        self.bert.embeddings.word_embeddings = None # Bỏ word_embedding mặc định
        
        # 3. BA LỚP LINEAR DỰ ĐOÁN
        self.fc_onset = nn.Linear(config.hidden_size, config.vocab_size)
        self.fc_rhyme = nn.Linear(config.hidden_size, config.vocab_size)
        self.fc_tone = nn.Linear(config.hidden_size, config.vocab_size)
        
        # 4. HÀM LOSS CHUNG
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
        self.init_weights()

    def forward(self, input_ids, attention_mask=None, labels=None):
        # input_ids: (B, L, 3)
        onset_embs = self.shared_embeddings(input_ids[:, :, 0]) # (B, L, 256)
        rhyme_embs = self.shared_embeddings(input_ids[:, :, 1]) # (B, L, 256)
        tone_embs  = self.shared_embeddings(input_ids[:, :, 2]) # (B, L, 256)
        
        # Nối lại thành 768
        inputs_embeds = torch.cat([onset_embs, rhyme_embs, tone_embs], dim=-1) # (B, L, 768)
        
        outputs = self.bert(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state # (B, L, 768)
        
        # Đi qua 3 FC Heads
        logit_onset = self.fc_onset(sequence_output) # (B, L, Vocab_size)
        logit_rhyme = self.fc_rhyme(sequence_output)
        logit_tone  = self.fc_tone(sequence_output) 
        
        logits = torch.stack([logit_onset, logit_rhyme, logit_tone], dim=2) # (B, L, 3, Vocab_size)
        
        loss = None
        if labels is not None:
            B, L, _, V = logits.shape
            # Duỗi tensor để tính 1 hàm loss duy nhất 
            active_logits = logits.view(-1, V) # (B * L * 3, Vocab_size)
            active_labels = labels.view(-1)    # (B * L * 3)
            loss = self.loss_fn(active_logits, active_labels)
            
        return {"loss": loss, "logits": logits}

# Khởi tạo mô hình
config = PhonemeBertConfig(
    vocab_size=VOCAB_SIZE, 
    num_hidden_layers=12, 
    num_attention_heads=12
)
model = PhonemeBertForMLM(config).to(device)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Section 4: Training Loop
Bắt đầu quá trình huấn luyện mô hình. Hàm tối ưu được sử dụng là AdamW.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

EPOCHS = 3
print("="*60)
print("Starting Phoneme-based BERT Pre-training")
print("="*60)

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch}/{EPOCHS}")
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs['loss']
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch} - Average Loss: {avg_loss:.4f}")

print("Training Complete!")

## Section 5: Save Model
Lưu trọng số mô hình đã huấn luyện thành công xuống thư mục `/kaggle/working` để tải về.

In [ ]:
output_dir = f"{BASE_DIR}/phoneme_bert_model"
os.makedirs(output_dir, exist_ok=True)

# Save model weights and config
torch.save(model.state_dict(), f"{output_dir}/pytorch_model.bin")
model.config.save_pretrained(output_dir)

print(f"Model saved successfully at {output_dir}")